In [83]:
import sys
import pandas as pd
from pathlib import Path
print(sys.executable)
print(pd.__version__)

c:\Users\caswv\projects\Investment-Portfolio-Analysis\.venv\Scripts\python.exe
3.0.5


In [84]:
DATA_PATH = Path("../data/raw/portfolio_20261031.xlsx")

print(DATA_PATH.resolve())
print(DATA_PATH.exists())

C:\Users\caswv\projects\Investment-Portfolio-Analysis\data\raw\portfolio_20261031.xlsx
True


In [85]:
quality_raw =   pd.read_excel(DATA_PATH, sheet_name="1) Kwaliteit", header=None)
quality_raw.shape

(102, 27)

In [86]:
header_mask = quality_raw[0] == "Ticker"
header_mask

0      False
1      False
2      False
3      False
4      False
       ...  
97     False
98     False
99     False
100    False
101    False
Name: 0, Length: 102, dtype: bool

In [87]:
quality_raw[header_mask]

,0,1,2,3,4,5,6,7,8,9,...,17,18,19,20,21,22,23,24,25,26
26,Ticker,Naam,ISIN,Type,Thema,Model,Weging pie,Weging portf.,Land,Beurs,...,CHF,CAD,Overig ontw.,Opkomend,Controle valuta,Hoofdvaluta,Div. rendement,Cijfer,Rating,Rol: waarom deze positie


In [88]:
quality_raw.index[header_mask]

RangeIndex(start=26, stop=27, step=1)

In [89]:
header_row = quality_raw.index[header_mask][0]
header_row

26

In [90]:
quality = pd.read_excel(DATA_PATH, sheet_name="1) Kwaliteit", header=header_row)
quality.shape

(75, 27)

In [91]:
quality.iloc[14:25, :8]

,Ticker,Naam,ISIN,Type,Thema,Model,Weging pie,Weging portf.
14,EW,Edwards Lifesciences,US28176E1082,Aandeel,Medtech,Kwaliteit,0.05,0.0135
15,SYK,Stryker,US8636671013,Aandeel,Medtech,Kwaliteit,0.05,0.0135
16,STMN,Straumann,CH1175448666,Aandeel,Medtech,Kwaliteit,0.05,0.0135
17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,Totaal,NaN,NaN,NaN,NaN,NaN,1.0,0.27
21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,Themaverdeling,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,Thema (sleutel),Omschrijving,Weging pie,Weging portf.,Aantal,NaN,NaN,NaN


In [92]:
empty_ticker_mask = quality["Ticker"].isna()
empty_ticker_mask.head(20)

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17     True
18     True
19     True
Name: Ticker, dtype: bool

In [93]:
first_empty = quality.index[empty_ticker_mask][0]
first_empty

np.int64(17)

In [94]:
quality_holdings = quality.iloc[:first_empty]
quality_holdings.shape

(17, 27)

In [95]:
quality_holdings.tail(3)

,Ticker,Naam,ISIN,Type,Thema,Model,Weging pie,Weging portf.,Land,Beurs,...,CHF,CAD,Overig ontw.,Opkomend,Controle valuta,Hoofdvaluta,Div. rendement,Cijfer,Rating,Rol: waarom deze positie
14,EW,Edwards Lifesciences,US28176E1082,Aandeel,Medtech,Kwaliteit,0.05,0.0135,VS,Xetra,...,0.0,0.0,0.0,0.0,OK,USD,0.0000,8.4,Buy,Leider in TAVR; ander verdienmodel dan ISRG en...
15,SYK,Stryker,US8636671013,Aandeel,Medtech,Kwaliteit,0.05,0.0135,VS,NYSE,...,0.0,0.0,0.0,0.0,OK,USD,0.0121,8.1,Buy,Orthopedie en medsurg tegen een historisch lag...
16,STMN,Straumann,CH1175448666,Aandeel,Medtech,Kwaliteit,0.05,0.0135,Zwitserland,SIX,...,1.0,0.0,0.0,0.0,OK,CHF,0.0103,8.1,Buy,Leider in premium tandimplantaten; CHF; gekoze...


In [96]:
old_weights = pd.to_numeric(quality["Weging pie"], errors="coerce")
old_filter = old_weights.notna()

print(old_filter.sum())
print(old_weights[old_filter].sum())

36
132.7


In [97]:
quality_holdings["Weging pie"].sum()

1.0000000000000004

In [98]:
def pie_weight_sum(holdings):
    """Return the sum of the pie weights of a holdings table."""
    return holdings["Weging pie"].sum()


pie_weight_sum(quality_holdings)

1.0000000000000004

In [99]:
def read_table(path, sheet_name, header_label):
    """Read a table from a sheet whose header row starts with header_label.

    Finds the header row (first column equals header_label), re-reads the
    sheet from that row and cuts the table off at the first empty cell in
    the first column.
    """
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    header_row = raw[raw[0] == header_label].index[0]

    table = pd.read_excel(path, sheet_name=sheet_name, header=header_row)

    first_col = table.columns[0]
    first_empty = table.index[table[first_col].isna()][0]
    return table.iloc[:first_empty]

In [100]:
read_holdings(DATA_PATH, "1) Kwaliteit").shape

(17, 27)

In [101]:
read_holdings(DATA_PATH, "3) AI Boom").shape

(8, 27)

In [102]:
quality.iloc[:first_empty + 1].tail(2)

,Ticker,Naam,ISIN,Type,Thema,Model,Weging pie,Weging portf.,Land,Beurs,...,CHF,CAD,Overig ontw.,Opkomend,Controle valuta,Hoofdvaluta,Div. rendement,Cijfer,Rating,Rol: waarom deze positie
16,STMN,Straumann,CH1175448666,Aandeel,Medtech,Kwaliteit,0.05,0.0135,Zwitserland,SIX,...,1.0,0.0,0.0,0.0,OK,CHF,0.0103,8.1,Buy,Leider in premium tandimplantaten; CHF; gekoze...
17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [103]:
numbers = [1, 2, 3]
squares = []
for n in numbers:
    squares.append(n ** 2)

squares

[1, 4, 9]

In [104]:
PIE_SHEETS = [
    "1) Kwaliteit",
    "2) Thematisch",
    "3) AI Boom",
    "4) Energie",
    "5) Mijnbouw",
    "6) Multipolaire Wereldorde",
]

In [105]:
frames = []

holdings = read_holdings(DATA_PATH, "1) Kwaliteit")
holdings["pie_sheet"] = "1) Kwaliteit"
frames.append(holdings)

len(frames)

1

In [106]:
holdings[["Ticker", "pie_sheet"]].head(3)

,Ticker,pie_sheet
0,V,1) Kwaliteit
1,MCO,1) Kwaliteit
2,LLY,1) Kwaliteit


In [107]:
frames = []
for sheet_name in PIE_SHEETS:
    holdings = read_table(DATA_PATH, sheet_name, "Ticker")
    holdings = holdings.rename(columns=CATEGORY_COLUMNS)
    holdings["pie_sheet"] = sheet_name
    holdings["pie"] = sheet_name.split(") ", maxsplit=1)[1]
    frames.append(holdings)

len(frames)

6

In [108]:
all_holdings = pd.concat(frames, ignore_index=True)
all_holdings.shape

(55, 29)

In [109]:
all_holdings.columns

Index(['Ticker', 'Naam', 'ISIN', 'Type', 'category', 'Model', 'Weging pie',
       'Weging portf.', 'Land', 'Beurs', 'Not. valuta', 'TER', 'Replicatie',
       'Uitkering', 'USD', 'EUR', 'GBP', 'CHF', 'CAD', 'Overig ontw.',
       'Opkomend', 'Controle valuta', 'Hoofdvaluta', 'Div. rendement',
       'Cijfer', 'Rating', 'Rol: waarom deze positie', 'pie_sheet', 'pie'],
      dtype='str')

In [110]:
holdings[["Ticker", "pie_sheet"]].head(3)

,Ticker,pie_sheet
0,EPIN,6) Multipolaire Wereldorde
1,AMEL,6) Multipolaire Wereldorde
2,CHN,6) Multipolaire Wereldorde


In [111]:
all_holdings = pd.concat(frames, ignore_index=True)
all_holdings.shape

(55, 29)

In [112]:
all_holdings["pie_sheet"].value_counts()

pie_sheet
1) Kwaliteit                  17
4) Energie                    10
5) Mijnbouw                    9
3) AI Boom                     8
6) Multipolaire Wereldorde     6
2) Thematisch                  5
Name: count, dtype: int64

In [113]:
CATEGORY_COLUMNS = {"Thema": "category", "Ketenlaag": "category", "Regio": "category"}

In [114]:
COLUMN_NAMES = {
    "Ticker": "ticker",
    "Naam": "name",
    "ISIN": "isin",
    "Type": "asset_type",
    "Model": "model",
    "Weging pie": "weight_in_pie",
    "Weging portf.": "weight_in_portfolio",
    "Land": "country",
    "Beurs": "exchange",
    "Not. valuta": "listing_currency",
    "TER": "ter",
    "Replicatie": "replication",
    "Uitkering": "distribution",
    "USD": "econ_usd",
    "EUR": "econ_eur",
    "GBP": "econ_gbp",
    "CHF": "econ_chf",
    "CAD": "econ_cad",
    "Overig ontw.": "econ_other_dm",
    "Opkomend": "econ_em",
    "Controle valuta": "econ_check",
    "Hoofdvaluta": "main_currency",
    "Div. rendement": "dividend_yield",
    "Cijfer": "score",
    "Rating": "rating",
    "Rol: waarom deze positie": "role",
}
all_holdings = all_holdings.rename(columns=COLUMN_NAMES)
list(all_holdings.columns)

['ticker',
 'name',
 'isin',
 'asset_type',
 'category',
 'model',
 'weight_in_pie',
 'weight_in_portfolio',
 'country',
 'exchange',
 'listing_currency',
 'ter',
 'replication',
 'distribution',
 'econ_usd',
 'econ_eur',
 'econ_gbp',
 'econ_chf',
 'econ_cad',
 'econ_other_dm',
 'econ_em',
 'econ_check',
 'main_currency',
 'dividend_yield',
 'score',
 'rating',
 'role',
 'pie_sheet',
 'pie']

In [115]:
toy = pd.DataFrame({
    "pie": ["A", "A", "B", "B", "B"],
    "weight": [0.6, 0.4, 0.5, 0.3, 0.2],
})
toy.groupby("pie")["weight"].sum()

pie
A    1.0
B    1.0
Name: weight, dtype: float64

In [116]:
pie_sums = all_holdings.groupby("pie_sheet")["weight_in_pie"].sum()
pie_sums

pie_sheet
1) Kwaliteit                  1.0
2) Thematisch                 1.0
3) AI Boom                    1.0
4) Energie                    1.0
5) Mijnbouw                   1.0
6) Multipolaire Wereldorde    1.0
Name: weight_in_pie, dtype: object

In [117]:
NUMERIC_COLUMNS = [
    "weight_in_pie",
    "weight_in_portfolio",
    "ter",
    "econ_usd",
    "econ_eur",
    "econ_gbp",
    "econ_chf",
    "econ_cad",
    "econ_other_dm",
    "econ_em",
    "dividend_yield",
    "score",
]

In [118]:
for col in NUMERIC_COLUMNS:
    all_holdings[col] = pd.to_numeric(all_holdings[col])

In [119]:
all_holdings[NUMERIC_COLUMNS].dtypes

weight_in_pie          float64
weight_in_portfolio    float64
ter                    float64
econ_usd               float64
econ_eur               float64
econ_gbp               float64
econ_chf               float64
econ_cad               float64
econ_other_dm          float64
econ_em                float64
dividend_yield         float64
score                  float64
dtype: object

In [120]:
pie_sums = all_holdings.groupby("pie_sheet")["weight_in_pie"].sum()
pie_sums

pie_sheet
1) Kwaliteit                  1.0
2) Thematisch                 1.0
3) AI Boom                    1.0
4) Energie                    1.0
5) Mijnbouw                   1.0
6) Multipolaire Wereldorde    1.0
Name: weight_in_pie, dtype: float64

In [121]:
TOLERANCE = 1e-9

pie_check = (pie_sums - 1).abs() < TOLERANCE
pie_check.all()

np.True_

In [122]:
total_weight = all_holdings["weight_in_portfolio"].sum()
abs(total_weight - 1) < TOLERANCE

np.True_

In [123]:
all_holdings.shape

(55, 29)

In [130]:
targets = read_table(DATA_PATH, "Portfolio", "Pie")
targets = targets[targets["Pie"] != "Totaal"]
targets = targets.rename(columns={"Pie": "pie", "Doelgewicht": "target_weight", "Blok": "block", "Posities": "n_positions"})
targets["target_weight"] = pd.to_numeric(targets["target_weight"])
targets["n_positions"] = pd.to_numeric(targets["n_positions"])
targets[["pie", "target_weight", "block", "n_positions"]]

,pie,target_weight,block,n_positions
0,Kwaliteit,0.27,A,17.0
1,Thematisch,0.13,A,5.0
2,AI Boom,0.10,A,8.0
3,Energie,0.18,B,10.0
4,Mijnbouw,0.17,B,9.0
5,Multipolaire Wereldorde,0.15,B,6.0


In [129]:
actual = all_holdings.groupby("pie")["weight_in_portfolio"].sum()
target = targets.set_index("pie")["target_weight"]

target_check = (actual - target).abs() < TOLERANCE
target_check.all()

np.True_

In [131]:
ECON_COLUMNS = ["econ_usd", "econ_eur", "econ_gbp", "econ_chf", "econ_cad", "econ_other_dm", "econ_em"]
row_sums = all_holdings[ECON_COLUMNS].sum(axis=1)

In [132]:
actual_counts = all_holdings.groupby("pie")["ticker"].count()
target_counts = targets.set_index("pie")["n_positions"]

In [134]:
checks = {
    "total weight = 1": abs(total_weight - 1) < TOLERANCE,
    "pie weights sum to 1": pie_check.all(),
    "pie weights match targets": target_check.all(),
      "position counts match Portfolio": ((actual_counts - target_counts) == 0).all(),
    "no empty tickers": all_holdings["ticker"].notna().all(),
    "no empty ISINs": all_holdings["isin"].notna().all(),
    "unique tickers": not all_holdings["ticker"].duplicated().any(),
    "unique ISINs": not all_holdings["isin"].duplicated().any(),
    "currency split sums to 1 per row": ((row_sums - 1).abs() < TOLERANCE).all(),
}
validation = pd.Series(checks)
validation

total weight = 1                    True
pie weights sum to 1                True
pie weights match targets           True
position counts match Portfolio     True
no empty tickers                    True
no empty ISINs                      True
unique tickers                      True
unique ISINs                        True
currency split sums to 1 per row    True
dtype: bool

In [135]:
assert validation.all(), validation[~validation]